## Import Required Libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType 

In [0]:
%run  /Workspace/Shopvista_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog","abc","catalog")
dbutils.widgets.text("data_source", "xyz", "data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
print(catalog, data_source)

In [0]:
base_path = f"s3://shopvista-sv/order_returns/"
landing_path = f"{base_path}/landing/"
processed_path =f"{base_path}/processed/"

print("Base_path:", base_path)
print("landing_path", landing_path)
print("processed_path", processed_path)

# Define the table
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table =f"{catalog}.{gold_schema}.{data_source}"

In [0]:
df = (spark.read
        .options(header=True, inferSchema=True)
        .csv(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_path")
)

In [0]:
display(df.limit(5))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )